# Notebook 8 – Text Classification Model

## Concept
Text classification assigns a label to a piece of text. A pre-trained model (here, DistilBERT fine-tuned on SST-2 for sentiment) already knows English, so we only need to: load the **tokenizer** (text → token IDs), load the **model** (token IDs → logits), turn logits into **probabilities** with softmax, and read the top label as the prediction.

Pipeline: `text → tokenizer → model → logits → softmax → label + confidence`

## Why it's needed
- No need to train from scratch: a pre-trained model gives a strong baseline in minutes.
- Understanding each step (not just `pipeline()`) lets you debug wrong predictions.
- Confidence scores tell you when to trust the model and when to send the text to a human.

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Load tokenizer and model
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

# 2. Prepare input
texts = [
    "I love this phone, the battery lasts all day!",           # Positive
    "The delivery was late and the product was damaged.",      # Negative
    "The meeting is scheduled for 3 PM tomorrow.",             # Neutral
    "The movie was not bad, but I won't watch it again.",      # Ambiguous
    "Oh great, another update that breaks everything.",        # Sarcasm
]

# 3. Tokenize
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
print("Input shape:", inputs["input_ids"].shape)

# 4. Run inference
with torch.no_grad():
    outputs = model(**inputs)

# 5. Process output (logits -> probabilities)
probs = torch.softmax(outputs.logits, dim=1)

# 6. Display prediction + confidence
for text, p in zip(texts, probs):
    idx = p.argmax().item()
    label = model.config.id2label[idx]
    print(f"{label:8} ({p[idx]:.2%}) | {text}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Input shape: torch.Size([5, 17])
POSITIVE (99.94%) | I love this phone, the battery lasts all day!
NEGATIVE (99.98%) | The delivery was late and the product was damaged.
POSITIVE (98.21%) | The meeting is scheduled for 3 PM tomorrow.
NEGATIVE (88.58%) | The movie was not bad, but I won't watch it again.
POSITIVE (99.84%) | Oh great, another update that breaks everything.


## What this example tells us
- The model is **binary** (POSITIVE / NEGATIVE). It has no neutral class, so neutral text is *forced* into one of the two labels, often with surprisingly high confidence.
- Softmax confidence means "how much the model prefers one label over the other", **not** "how likely it is to be correct".
- Sarcasm, negation (`not bad`) and mixed sentiment are the usual failure cases. Check the ANALYSIS section printed above for which of your inputs were wrong, forced, or low-confidence.

## AI/ML Example
Using a pre-trained classifier as a baseline before fine-tuning. If it fails on your domain's neutral or sarcastic text, that is the evidence you need to fine-tune on a 3-class dataset (negative / neutral / positive).

## Business Example
A support team auto-tags customer reviews. Clear praise and complaints are tagged automatically; reviews with low confidence or mixed sentiment are sent to a human agent instead of being mis-tagged.

## AI/ML Engineer Use Case
Build a confidence threshold (for example, route anything below 0.80 to review), and keep a small "hard cases" test set (sarcasm, negation, neutral, mixed) to re-run whenever the model or data changes.